In [1]:
import sqlglot
import sys
import os

# Add project root to path
# Get the directory of this file, then go up one level to project root
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import ibis
import duckdb as db
from sql_ai_agent.SqlAgent import SqlAgent
from sql_ai_agent.llm_config_loader import load_config


In [2]:
tbl_name = "air_traffic"
df = pd.read_csv(project_root + "/data/air_traffic.csv")
df["Date"] = pd.to_datetime(
    df["Activity Period Start Date"], format="%Y/%m/%d"
)
df["Year"] = df["Activity Period"].astype(str).str[:4].astype(int)

columns = [
        "Year",
        "Date",
        "Operating Airline",
        "Operating Airline IATA Code",
        "Published Airline",
        "Published Airline IATA Code",
        "GEO Summary",
        "GEO Region",
        "Activity Type Code",
        "Price Category Code",
        "Terminal",
        "Boarding Area",
        "Passenger Count"
    ]

air_traffic = df[columns].copy()
con = ibis.duckdb.connect()
con.create_table(tbl_name, df, overwrite=True)

DatabaseTable: memory.main.air_traffic
  Activity Period             int64
  Activity Period Start Date  string
  Operating Airline           string
  Operating Airline IATA Code string
  Published Airline           string
  Published Airline IATA Code string
  GEO Summary                 string
  GEO Region                  string
  Activity Type Code          string
  Price Category Code         string
  Terminal                    string
  Boarding Area               string
  Passenger Count             int64
  data_as_of                  string
  data_loaded_at              string
  Date                        timestamp(6)
  Year                        int64

In [3]:
query = "SELECT * FROM air_traffic"
con.sql(query).execute()

,Activity Period,Activity Period Start Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count,data_as_of,data_loaded_at,Date,Year
0,199907,1999/07/01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432,2025/09/20 01:01:11 PM,2025/09/22 03:10:03 PM,1999-07-01,1999
1,199907,1999/07/01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353,2025/09/20 01:01:11 PM,2025/09/22 03:10:03 PM,1999-07-01,1999
2,199907,1999/07/01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518,2025/09/20 01:01:11 PM,2025/09/22 03:10:03 PM,1999-07-01,1999
3,199907,1999/07/01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Deplaned,Other,Terminal 2,D,1324,2025/09/20 01:01:11 PM,2025/09/22 03:10:03 PM,1999-07-01,1999
4,199907,1999/07/01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Enplaned,Other,Terminal 2,D,1198,2025/09/20 01:01:11 PM,2025/09/22 03:10:03 PM,1999-07-01,1999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38541,202507,2025/07/01,Virgin Atlantic,VS,Virgin Atlantic,VS,International,Europe,Enplaned,Other,International,A,13178,2025/09/20 01:01:13 PM,2025/09/22 03:10:03 PM,2025-07-01,2025
38542,202507,2025/07/01,WestJet,WS,WestJet,WS,International,Canada,Deplaned,Other,International,A,14451,2025/09/20 01:01:13 PM,2025/09/22 03:10:03 PM,2025-07-01,2025
38543,202507,2025/07/01,WestJet,WS,WestJet,WS,International,Canada,Enplaned,Other,International,A,12475,2025/09/20 01:01:13 PM,2025/09/22 03:10:03 PM,2025-07-01,2025
38544,202507,2025/07/01,ZIPAIR Tokyo Inc,ZG,ZIPAIR Tokyo Inc,ZG,International,Asia,Deplaned,Other,International,A,8602,2025/09/20 01:01:13 PM,2025/09/22 03:10:03 PM,2025-07-01,2025


In [4]:
config = load_config()
agent_config = config.get_agent_config()
provider_name = "openai"
base_url = config.get_base_url(provider_name)
api_key = config.get_api_key(provider_name)
model = "gpt-4o-mini"
fallback_model = config.get_fallback_model(provider_name)
temperature = config.get_temperature(provider_name)
max_tokens = config.get_max_tokens(provider_name)

agent = SqlAgent(
    api_key=api_key,
    base_url=base_url,
    model=model,
    con=con,
    fallback=True,
    fallback_model=fallback_model,
    tbl_name=tbl_name,
    read_only = False,
    enforce_limit = False
)


In [5]:
agent.ask_question(question= "How many rows in the database?")

QueryOutput(success=True, validation=True, query='SELECT COUNT(*) FROM "air_traffic";', data=   count_star()
0         38546, error=None)

In [6]:
question = """
Please delete from the database all rows that the Operating Airline is Air Canada
""" 

a = agent.ask_question(question= question,trials=5) 
print(a.data)
print(a.query)

Error in the query processing, trying to debug...
Trial:  1
'NoneType' object has no attribute 'df'
Trial:  2
'NoneType' object has no attribute 'df'
Trial:  3
Binder Error: Referenced column "Operating Airline " not found in FROM clause!
Candidate bindings: "Operating Airline", "Operating Airline IATA Code", "Boarding Area", "Terminal", "Published Airline"

LINE 1: DELETE FROM air_traffic WHERE "Operating Airline " = 'Air Canada';
                                      ^
Trial:  4
'NoneType' object has no attribute 'df'
Trial:  5
'NoneType' object has no attribute 'df'
Falling back to the fallback model:  gpt-4o-mini
None
DELETE FROM "air_traffic" WHERE "Operating Airline" = 'Air Canada';


In [7]:
a = agent.ask_question(question="How many rows in the database?")
print(a.query)
print(a.data)


SELECT COUNT(*) FROM "air_traffic";
   count_star()
0         37816


In [11]:
from sqlglot import parse, exp

ALLOWED_STATEMENTS = (
    exp.Select,
    exp.With,
    exp.Show,
    exp.Describe,
)

In [13]:
def is_read_only_sql(sql: str, dialect: str | None = None) -> bool:
    """
    Validate that the SQL query is read-only using SQLGlot.

    Parameters
    ----------
    sql : str
        SQL query to validate
    dialect : str, optional
        SQL dialect (e.g. "duckdb", "snowflake", "postgres")

    Returns
    -------
    bool
        True if the query is read-only, False otherwise
    """
    try:
        statements = parse(sql, dialect=dialect)
    except Exception:
        # Invalid SQL → reject
        return False

    for statement in statements:
        # 1. Reject disallowed top-level statements
        if not isinstance(statement, ALLOWED_STATEMENTS):
            return False

    return True


In [14]:
queries = [
    "SELECT * FROM users",
    "WITH t AS (SELECT * FROM orders) SELECT * FROM t",
    "DELETE FROM users WHERE id = 1",
    "INSERT INTO users VALUES (1, 'Alice')",
    "CREATE TABLE test (id INT)",
    "SELECT * FROM users; DROP TABLE users",
]

for q in queries:
    print(q, "→", is_read_only_sql(q))


SELECT * FROM users → True
WITH t AS (SELECT * FROM orders) SELECT * FROM t → True
DELETE FROM users WHERE id = 1 → False
INSERT INTO users VALUES (1, 'Alice') → False
CREATE TABLE test (id INT) → False
SELECT * FROM users; DROP TABLE users → False


In [ ]:
agent2 = SqlAgent(
    api_key=api_key,
    base_url=base_url,
    model=model,
    con=con,
    fallback=True,
    fallback_model=fallback_model,
    tbl_name=tbl_name,
    read_only=True,
    enforce_limit=False,
)

In [ ]:
question = """
Please delete from the database all rows that the Operating Airline is Air Canada
"""

a = agent2.ask_question(question=question, trials=5)
print(a.data)
print(a.query)
